In [1]:
import fitz 
import google.generativeai as genai
import os
import numpy as np

/home/zogning-abel/miniconda3/envs/ml/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.16) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/zogning-abel/miniconda3/envs/ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_72726/2197422166.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

h

In [3]:
# ==========================================
# 1. SETUP GEMINI API
# ==========================================
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel('gemini-2.5-flash')
EMBEDDING_MODEL = "gemini-embedding-001"

In [44]:
# ==========================================
# 2. CHUNKING
# ==========================================
def extract_and_chunk(pdf_path, chunk_size=200):
    """Extracts text from PDF and splits it into chunks of roughly N words."""
    print(f"Extracting and chunking: {pdf_path}")
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text("text") + "\n"
        
    words = text.split()
    chunks = [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]
    print(f"Created {len(chunks)} chunks.")
    return chunks

In [45]:
# ==========================================
# 3. GET EMBEDDINGS
# ==========================================
def get_embedding(text):
    """Converts a single string of text into a vector (list of floats)."""
    response = genai.embed_content(
        model=EMBEDDING_MODEL,
        content=text,
        task_type="retrieval_document"
    )
    return response['embedding']

In [46]:
# ==========================================
# 4. CREATE A VECTOR STORE (MAPPING TABLE)
# ==========================================
def create_vector_store(chunks):
    """
    Creates the 'mapping table'. 
    Returns a list of dictionaries containing the text chunk and its mathematical vector.
    """
    print("Converting chunks to vectors (Embedding Space)...")
    vector_store = []
    for index, chunk in enumerate(chunks):
        vector = get_embedding(chunk)
        vector_store.append({
            "id": index,
            "text": chunk,
            "vector": vector
        })
    print("Mapping table created successfully!")
    return vector_store

In [47]:
# ==========================================
# 5. COSINE SIMILARITY FUNCTION
# ==========================================
def cosine_similarity(vec1, vec2):
    """Calculates the mathematical distance between two vectors."""
    a = np.array(vec1)
    b = np.array(vec2)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [48]:
# ==========================================
# 6. RETRIEVE TOP K CHUNKS
# ==========================================
def retrieve_top_k_chunks(user_query, vector_store, k=3):
    """Converts the query to a vector, compares it to the table, and gets the top K matches."""
    print(f"\nSearching for chunks relevant to: '{user_query}'")
    
    # 1. Convert user query to vector
    query_vector = genai.embed_content(
        model=EMBEDDING_MODEL,
        content=user_query,
        task_type="retrieval_query"
    )['embedding']
    
    # 2. Compare query vector to every chunk vector in the mapping table
    scored_chunks = []
    for item in vector_store:
        similarity_score = cosine_similarity(query_vector, item["vector"])
        scored_chunks.append((similarity_score, item["text"]))
    
    # 3. Sort by highest score and get the top K results
    scored_chunks.sort(key=lambda x: x[0], reverse=True)
    top_chunks = [chunk[1] for chunk in scored_chunks[:k]]
    
    return top_chunks

In [49]:
# ==========================================
# 7. GENERATE FINAL ANSWER
# ==========================================
def generate_answer(user_query, retrieved_chunks):
    """Passes the chosen chunks and the user request to the LLM."""
    print("Generating final answer...\n")
    
    # Combine the retrieved chunks into one context block
    context = "\n\n---\n\n".join(retrieved_chunks)
    
    # Create the RAG prompt
    prompt = f"""
    You are a strict technical extraction assistant. Your sole purpose is to answer the user's question based STRICTLY and EXCLUSIVELY on the provided context.

    RULES:
    1. Answer ONLY what the user explicitly asks. Do not provide extra, unprompted information.
    2. Be extremely concise and direct.(e.g., do not say "Based on the document..." or "Here is the answer...").
    3. Rely ONLY on the provided context. Do NOT use any outside knowledge.
    4. Do not guess, infer, or hallucinate information.
    5. If the context does not contain the exact answer, you must output exactly: "I cannot find the answer in the provided document."

    CONTEXT:
    {context}
    
    USER QUESTION:
    {user_query}
    """
    
    response = model.generate_content(prompt)
    return response.text

In [50]:
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

In [51]:
SEPARATEUR_PDF = "data/PRE-SEPARATEUR.pdf" 
    
separateur_document_chunks = extract_and_chunk(SEPARATEUR_PDF)
separateur_mapping_table = create_vector_store(separateur_document_chunks)

Extracting and chunking: data/PRE-SEPARATEUR.pdf
Created 8 chunks.
Converting chunks to vectors (Embedding Space)...
Mapping table created successfully!


In [52]:
separateur_user_question = "Quelles sont les caractéristiques principales de ce pré-séparateur ?"

separateur_best_chunks = retrieve_top_k_chunks(separateur_user_question, separateur_mapping_table, k=2)
separateur_final_answer = generate_answer(separateur_user_question, separateur_best_chunks)

with open(f"{output_dir}/separateur_mapping_table.txt", "w") as f:
    for item in separateur_mapping_table:
        f.write(f"ID: {item['id']}\n")
        f.write(f"Text: {item['text']}\n")
        f.write(f"Vector: {item['vector']}\n")
        f.write("-" * 20 + "\n")
with open(f"{output_dir}/separateur_best_chunks.txt", "w") as f:
    for chunk in separateur_best_chunks:
        f.write(chunk + "\n")
with open(f"{output_dir}/separateur_final_answer.txt", "w") as f:
    f.write("USER QUESTION:\n")
    f.write(separateur_user_question + "\n")
    f.write("\nFINAL ANSWER:\n")
    f.write(separateur_final_answer)

print("-" * 50)
print("FINAL ANSWER:\n")
print(separateur_final_answer)


Searching for chunks relevant to: 'Quelles sont les caractéristiques principales de ce pré-séparateur ?'
Generating final answer...

--------------------------------------------------
FINAL ANSWER:

Il est conçu pour le captage de poussières lourdes et autres débris, produit en acier avec des roues, sans cartouche de filtration (pas pour liquides), a un poids de 14 kg, une capacité de 33 L, et des dimensions de 450 x 490 x 840 mm.


In [53]:
TUYAUX_PDF = "data/tuyaux.pdf"

tuyaux_document_chunks = extract_and_chunk(TUYAUX_PDF)
tuyaux_mapping_table = create_vector_store(tuyaux_document_chunks)

Extracting and chunking: data/tuyaux.pdf
Created 14 chunks.
Converting chunks to vectors (Embedding Space)...
Mapping table created successfully!


In [54]:
tuyaux_user_question = "Quels sont les différents types de tuyaux mentionnés dans le document, et quelles sont leurs caractéristiques (diamètre, matière, pression) ?"

tuyaux_best_chunks = retrieve_top_k_chunks(tuyaux_user_question, tuyaux_mapping_table, k=2)
tuyaux_final_answer = generate_answer(tuyaux_user_question, tuyaux_best_chunks)

with open(f"{output_dir}/tuyaux_mapping_table.txt", "w") as f:
    for item in tuyaux_mapping_table:
        f.write(f"ID: {item['id']}\n")
        f.write(f"Text: {item['text']}\n")
        f.write(f"Vector: {item['vector']}\n")
        f.write("-" * 20 + "\n")
with open(f"{output_dir}/tuyaux_best_chunks.txt", "w") as f:
    for chunk in tuyaux_best_chunks:
        f.write(chunk + "\n")
with open(f"{output_dir}/tuyaux_final_answer.txt", "w") as f:
    f.write("USER QUESTION:\n")
    f.write(tuyaux_user_question + "\n")
    f.write("\nFINAL ANSWER:\n")
    f.write(tuyaux_final_answer)
    
print("-" * 50)
print("FINAL ANSWER:\n")
print(tuyaux_final_answer)




Searching for chunks relevant to: 'Quels sont les différents types de tuyaux mentionnés dans le document, et quelles sont leurs caractéristiques (diamètre, matière, pression) ?'
Generating final answer...

--------------------------------------------------
FINAL ANSWER:

I cannot find the answer in the provided document.
